In [1]:

%load_ext ElasticNotebook

Enabled rmm statistics


In [2]:
import time

start_time = time.perf_counter()

In [3]:
%%RecordEvent
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
# %load_ext cudf.pandas
import numpy as np # linear algebra
import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt
import time
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
### cell 0 ###

data = pd.read_csv(
    "/home/dias-benchmarks/notebooks/josecode1/billionaires-statistics-2023/input/Billionaires Statistics Dataset.csv",
    index_col="rank",
)

In [5]:
### cell 1 ###

factor = 400  # 800
data = pd.concat([data] * factor, ignore_index=True)
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1056000 entries, 0 to 1055999
Data columns (total 34 columns):
 #   Column                                      Non-Null Count    Dtype  
---  ------                                      --------------    -----  
 0   finalWorth                                  1056000 non-null  int64  
 1   category                                    1056000 non-null  object 
 2   personName                                  1056000 non-null  object 
 3   age                                         1030000 non-null  float64
 4   country                                     1040800 non-null  object 
 5   city                                        1027200 non-null  object 
 6   source                                      1056000 non-null  object 
 7   industries                                  1056000 non-null  object 
 8   countryOfCitizenship                        1056000 non-null  object 
 9   organization                                130000 non-nu

In [6]:
### cell 2 ###

data.head(5)

,finalWorth,category,personName,age,country,city,source,industries,countryOfCitizenship,organization,...,cpi_change_country,gdp_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,life_expectancy_country,tax_revenue_country_country,total_tax_rate_country,population_country,latitude_country,longitude_country
0,211000,Fashion & Retail,Bernard Arnault & family,74.0,France,Paris,LVMH,Fashion & Retail,France,LVMH Moët Hennessy Louis Vuitton,...,1.1,"$2,715,518,274,227",65.6,102.5,82.5,24.2,60.7,67059887.0,46.227638,2.213749
1,180000,Automotive,Elon Musk,51.0,United States,Austin,"Tesla, SpaceX",Automotive,United States,Tesla,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
2,114000,Technology,Jeff Bezos,59.0,United States,Medina,Amazon,Technology,United States,Amazon,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
3,107000,Technology,Larry Ellison,78.0,United States,Lanai,Oracle,Technology,United States,Oracle,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
4,106000,Finance & Investments,Warren Buffett,92.0,United States,Omaha,Berkshire Hathaway,Finance & Investments,United States,Berkshire Hathaway Inc. (Cl A),...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891


In [7]:
### cell 3 ###

data.describe()

,finalWorth,age,birthYear,birthMonth,birthDay,cpi_country,cpi_change_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,life_expectancy_country,tax_revenue_country_country,total_tax_rate_country,population_country,latitude_country,longitude_country
count,1.056000e+06,1.030000e+06,1.025600e+06,1.025600e+06,1.025600e+06,982400.000000,982400.000000,983200.000000,983600.000000,983200.000000,982800.000000,983200.000000,9.904000e+05,990400.000000,990400.000000
mean,4.623788e+03,6.514019e+01,1.957183e+03,5.740250e+00,1.209984e+01,127.755204,4.364169,67.225671,102.858520,78.122823,12.546235,43.963344,5.102053e+08,34.903592,12.583156
std,9.832383e+03,1.325553e+01,1.327993e+01,3.709363e+00,9.916946e+00,26.447579,3.623027,21.339095,4.710022,3.729342,5.367535,12.142831,5.541331e+08,17.000071,86.745511
min,1.000000e+03,1.800000e+01,1.921000e+03,1.000000e+00,1.000000e+00,99.550000,-1.900000,4.000000,84.700000,54.300000,0.100000,9.900000,3.801900e+04,-40.900557,-106.346771
25%,1.500000e+03,5.600000e+01,1.948000e+03,2.000000e+00,1.000000e+00,117.240000,1.700000,50.600000,100.200000,77.000000,9.600000,36.600000,6.683440e+07,35.861660,-95.712891
50%,2.300000e+03,6.500000e+01,1.957000e+03,6.000000e+00,1.100000e+01,117.240000,2.900000,65.600000,101.800000,78.500000,9.600000,41.200000,3.282395e+08,37.090240,10.451526
75%,4.200000e+03,7.500000e+01,1.966000e+03,9.000000e+00,2.100000e+01,125.080000,7.500000,88.200000,102.600000,80.900000,12.800000,59.100000,1.366418e+09,40.463667,104.195397
max,2.110000e+05,1.010000e+02,2.004000e+03,1.200000e+01,3.100000e+01,288.570000,53.500000,136.600000,142.100000,84.200000,37.200000,106.300000,1.397715e+09,61.924110,174.885971


In [8]:
### cell 4 ###

country_names = data[
    "country"
].value_counts()  # List of how many billionaires there are in the country

In [9]:
### cell 5 ###

country_names

country
United States               301600
China                       209200
India                        62800
Germany                      40800
United Kingdom               32800
                             ...  
Turks and Caicos Islands       400
Tanzania                       400
Bahrain                        400
Andorra                        400
Armenia                        400
Name: count, Length: 78, dtype: int64

> > > > >  ****10 countries with the most billionaires****

In [10]:
### cell 6 ###

data_100 = data.loc[:100, ["finalWorth", "category", "country"]]

In [11]:
### cell 7 ###

data_100_category = data_100["category"].value_counts()

> > > > > ****Most Richest 100 person in the world and their categories****

In [12]:
### cell 8 ###

data_usa = data[
    data["country"] == "United States"
]  # We focus on billionaires based in United States

In [13]:
### cell 9 ###

data_usa_category = data_usa["category"].value_counts()
data_usa_category

category
Finance & Investments         76000
Technology                    56400
Food & Beverage               29600
Fashion & Retail              25200
Real Estate                   20400
Media & Entertainment         14800
Sports                        14000
Energy                        14000
Healthcare                    13600
Manufacturing                 11200
Service                        7600
Diversified                    5200
Automotive                     4800
Gambling & Casinos             2400
Telecom                        2000
Construction & Engineering     2000
Logistics                      2000
Metals & Mining                 400
Name: count, dtype: int64

> > > > > ****All Billionaires in the USA and their categories****

In [14]:
### cell 10 ###

data_usa_city = data_usa["city"].value_counts()
print(data_usa_city)
data_usa_city.info()

city
New York          39600
San Francisco     14800
Los Angeles       13600
Palm Beach         8400
Dallas             8000
                  ...  
Gallatin            400
Jupiter Island      400
San Carlos          400
Arcadia             400
Pottsville          400
Name: count, Length: 268, dtype: int64
<class 'pandas.core.series.Series'>
Index: 268 entries, New York to Pottsville
Series name: count
Non-Null Count  Dtype
--------------  -----
268 non-null    int64
dtypes: int64(1)
memory usage: 4.2+ KB


268 city exist, I'll try .head(20)

> > > > > **The 20 cities has most billionaires in usa**

In [15]:
### cell 11 ###

data.head(5)

,finalWorth,category,personName,age,country,city,source,industries,countryOfCitizenship,organization,...,cpi_change_country,gdp_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,life_expectancy_country,tax_revenue_country_country,total_tax_rate_country,population_country,latitude_country,longitude_country
0,211000,Fashion & Retail,Bernard Arnault & family,74.0,France,Paris,LVMH,Fashion & Retail,France,LVMH Moët Hennessy Louis Vuitton,...,1.1,"$2,715,518,274,227",65.6,102.5,82.5,24.2,60.7,67059887.0,46.227638,2.213749
1,180000,Automotive,Elon Musk,51.0,United States,Austin,"Tesla, SpaceX",Automotive,United States,Tesla,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
2,114000,Technology,Jeff Bezos,59.0,United States,Medina,Amazon,Technology,United States,Amazon,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
3,107000,Technology,Larry Ellison,78.0,United States,Lanai,Oracle,Technology,United States,Oracle,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
4,106000,Finance & Investments,Warren Buffett,92.0,United States,Omaha,Berkshire Hathaway,Finance & Investments,United States,Berkshire Hathaway Inc. (Cl A),...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891


In [16]:
### cell 12 ###

category_list = list(data.category.unique())
category_list

['Fashion & Retail',
 'Automotive',
 'Technology',
 'Finance & Investments',
 'Media & Entertainment',
 'Telecom',
 'Diversified',
 'Food & Beverage',
 'Logistics',
 'Gambling & Casinos',
 'Manufacturing',
 'Real Estate',
 'Metals & Mining',
 'Energy',
 'Healthcare',
 'Service',
 'Construction & Engineering',
 'Sports']

In [17]:
### cell 13 ###

data.finalWorth = data.finalWorth.astype(float)
data2 = data.groupby("category", as_index=False)["finalWorth"].mean()
data2.columns = ["category_list", "worth_average"]
new_data = data2.sort_values(by="worth_average", ascending=False).reset_index(drop=True)

In [18]:
### cell 14 ###

data3 = data.dropna(axis=0, how="any", subset=data.columns)

In [19]:
### cell 15 ###

data3.head()

,finalWorth,category,personName,age,country,city,source,industries,countryOfCitizenship,organization,...,cpi_change_country,gdp_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,life_expectancy_country,tax_revenue_country_country,total_tax_rate_country,population_country,latitude_country,longitude_country
1,180000.0,Automotive,Elon Musk,51.0,United States,Austin,"Tesla, SpaceX",Automotive,United States,Tesla,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.09024,-95.712891
2,114000.0,Technology,Jeff Bezos,59.0,United States,Medina,Amazon,Technology,United States,Amazon,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.09024,-95.712891
3,107000.0,Technology,Larry Ellison,78.0,United States,Lanai,Oracle,Technology,United States,Oracle,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.09024,-95.712891
4,106000.0,Finance & Investments,Warren Buffett,92.0,United States,Omaha,Berkshire Hathaway,Finance & Investments,United States,Berkshire Hathaway Inc. (Cl A),...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.09024,-95.712891
5,104000.0,Technology,Bill Gates,67.0,United States,Medina,Microsoft,Technology,United States,Bill & Melinda Gates Foundation,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.09024,-95.712891


In [20]:
### cell 16 ###

data3.info()  # in the new df we have 238 rows for each column

<class 'pandas.core.frame.DataFrame'>
Index: 95200 entries, 1 to 1055968
Data columns (total 34 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   finalWorth                                  95200 non-null  float64
 1   category                                    95200 non-null  object 
 2   personName                                  95200 non-null  object 
 3   age                                         95200 non-null  float64
 4   country                                     95200 non-null  object 
 5   city                                        95200 non-null  object 
 6   source                                      95200 non-null  object 
 7   industries                                  95200 non-null  object 
 8   countryOfCitizenship                        95200 non-null  object 
 9   organization                                95200 non-null  object 
 10  selfMade     

In [21]:
### cell 17 ###

data3.countryOfCitizenship.unique()  # and 238 billionaires in usa, others deleted.

array(['United States', 'Canada', 'Germany', 'Israel', 'Norway',
       'Ireland', 'Turkey', 'France'], dtype=object)

In [22]:
end_time = time.perf_counter()
print(end_time - start_time)

3.782047912998678
